<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 3 精华 — Research Agent + MCP

**一句话定位**:**Model Context Protocol(MCP)是 2024 末 Anthropic 推的工具接入标准协议** —— 工具一处写,任何 LLM/agent 都能用,跨语言、跨进程、跨组织。本节学怎么把 MCP server 接到 LangGraph agent。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 上一节我们已经能给 agent 加工具了,为啥还要 MCP?**

上一节用 `@tool` 装饰自定义 Python 函数。问题:

| 痛点 | 表现 |
|------|------|
| **工具跟代码耦合** | 工具改了,agent 代码也得改 |
| **跨语言难复用** | Python 写的工具,JS 项目用不了 |
| **每个团队各写各的** | 同样的「**查 Slack**」工具,5 个团队写 5 遍 |
| **第三方集成难做** | 公司 API 团队怎么给 agent 暴露能力? |

**MCP 的答案**:**把工具变成独立的 server,通过协议(MCP)暴露**。任何懂 MCP 的 client 都能用。

→ 这就是 **工具的「**标准化**」时刻**(类似 HTTP 之于 web、JDBC 之于数据库)。2026 年,**主流 SaaS 都在出官方 MCP server**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧩 MCP 架构 — Client-Server 模型

</div>

**核心组件 3 件套**:

```
┌──────────────────┐        ┌──────────────────┐        ┌──────────────────┐
│  LangGraph Agent │  ◀────▶│ MultiServerMCP   │  ◀────▶│  MCP Server      │
│  (LLM 决策 + 工具) │        │  Client(adapter) │        │  (Node / Python..)│
└──────────────────┘        └──────────────────┘        └──────────────────┘
        ↑                              ↑                          ↑
        │                              │                          │
      用 LangChain                  把 MCP 协议             实际跑工具
      tool 接口                     转 LangChain 格式       (读文件、调 API...)
```

| 角色 | 谁是 | 干啥 |
|------|------|------|
| **Client** | `MultiServerMCPClient`(LangChain MCP Adapter) | 跟 server 通信、协议转换 |
| **Server** | 第三方进程(Node / Python / Go...) | **提供工具 + 执行操作** |
| **协议** | MCP(基于 JSON-RPC) | client 和 server 怎么对话 |

**关键洞察**:**LangChain 不需要懂每个工具的细节** —— 只需要懂 MCP 协议,**就能用上所有 MCP server 提供的工具**。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🔀 两种 Transport — stdio vs HTTP

</div>

MCP server 跟 client 怎么通信?两种方式,**对应两种部署场景**:

| 维度 | **stdio**(本地) | **HTTP**(远程) |
|------|------------------|------------------|
| **通信方式** | stdin / stdout pipes | HTTP / WebSocket |
| **server 位置** | **本地子进程** | **远程服务器** |
| **client 启动** | 用配置里的 `command` **fork 一个子进程** | 直接 HTTP 连过去 |
| **速度** | ⚡ 快(进程内通信) | 🐢 慢(网络往返) |
| **安全** | 🔒 进程隔离,本地受控 | 🔐 需要 auth + 信任 |
| **适用** | 文件操作、私有工具 | SaaS、跨组织集成 |

### 🅰️ stdio 配置示例

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">mcp_config = {
    "filesystem": {                                    # server 名(任意标签)
        "command": "npx",                              # 启动命令
        "args": ["-y", "@modelcontextprotocol/server-filesystem", "/path/to/docs"],
        <span style="background:#3d3414; color:#fde047; padding:0 2px;">"transport": "stdio"</span>
    }
}
</code></pre>

### 🅱️ HTTP 配置示例(远程)

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">mcp_config = {
    "asana": {
        "url": "https://mcp.asana.com/sse",
        <span style="background:#3d3414; color:#fde047; padding:0 2px;">"transport": "http"</span>,
        "headers": {
            "Authorization": "Bearer your-token-here"
        }
    }
}
</code></pre>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 已支持远程 MCP 的大厂(部分)**

| 公司 | 服务 | URL 格式 |
|------|------|---------|
| **Anthropic** | Claude API 工具 | `https://mcp.anthropic.com/...` |
| **Asana** | 项目管理 | `https://mcp.asana.com/sse` |
| **Cloudflare** | 边缘服务、DNS、CDN | `https://mcp.cloudflare.com/sse` |
| **PayPal** | 支付、订单 | `https://mcp.paypal.com/sse` |
| **Zapier** | 自动化(数千集成) | `https://mcp.zapier.com/sse` |
| **Linear** | 工单管理 | `https://mcp.linear.app/sse` |

→ **不用自己接 OAuth、不用读 API 文档** —— 把 MCP URL 给 agent 就完事。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ⚡ 为啥 MCP 必须 async — 架构层原因

</div>

**用 MCP 时,你 *必须* 用 async 方法**(`await tool.ainvoke(...)`,**不能** `tool.invoke(...)`)。

这不是设计选择,是 **MCP 协议天然异步**:

| 因素 | 含义 |
|------|------|
| **server 通信** | JSON-RPC over stdio/http,网络/IPC,**延迟不确定** |
| **工具操作本身** | 文件 I/O、网络请求、DB 查询 —— 都是 I/O 密集 |
| **并发** | 多个工具可以 **同时** 跑,async 才能利用这点 |
| **server 是子进程** | 走 pipes 通信,**阻塞 sync 调用会冻住整个进程** |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 LangChain MCP Adapters 是 async-only 设计**

- 工具被包装成 **`async StructuredTools`**
- **sync 调用故意不实现** —— 强制你用 `await`
- 保证所有 MCP 工具行为 **一致**(不会有的 sync 有的 async,易混淆)

**实战写法**:

```python
tools = await client.get_tools()         # 异步拿工具列表
result = await tool.ainvoke(args)         # 异步执行
```

**对应的 graph node 也必须是 async**:

```python
async def tool_node(state):    # ← 注意 async def
    coros = [tool.ainvoke(args) for ...]
    results = await asyncio.gather(*coros)   # 并发执行!
    ...
```

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ Filesystem MCP Server — 本节实例

</div>

本节用 [**Filesystem MCP Server**](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem) 演示。

**它干啥**:**带细粒度权限管理的安全本地文件系统访问**。

### 📋 提供的工具(分 3 类)

| 类别 | 工具 |
|------|------|
| **文件操作** | `read_file` · `write_file` · `edit_file` · `read_multiple_files` |
| **目录管理** | `create_directory` · `list_directory` · `move_file` |
| **搜索发现** | `search_files` · `get_file_info` · `list_allowed_directories` |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 为啥选这个做演示?**

因为它 **简单、安全、本地可跑**:

- 只能访问 **配置里允许的目录**(`/path/to/docs`)
- 不需要 API key / OAuth
- `npx` 一行就能跑(`npx -y @modelcontextprotocol/server-filesystem`)

**学完这个再去看其他 MCP server**(Slack、Linear、Asana)就轻车熟路。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎬 完整数据流 — 一次工具调用的全过程

</div>

假设 LLM 决定调 `read_file(path="docs/report.md")`:

```
🅰️ 启动时(一次性):
  1. client = MultiServerMCPClient(mcp_config)
  2. client 用 npx 启动 filesystem server(子进程)
  3. tools = await client.get_tools()
     ↓ 内部做了:
     - client 问 server: 「你有啥工具?」
     - server 返回工具元数据(name, description, params, types)
     - client 把元数据 转 LangChain 格式
  4. model_with_tools = model.bind_tools(tools)

🅱️ 用户提问后(每次循环):
  5. LLM 看 state,决定调 read_file
  6. tool_call = {"name": "read_file", "args": {"path": "docs/report.md"}}
  7. 找到对应 tool 对象 → await tool.ainvoke(tool_call["args"])
     ↓ 内部做了:
     - LangChain MCP Adapter 把请求转成 MCP JSON-RPC 格式
     - 通过 stdio pipe 发给 filesystem server
     - server 真的读了文件
     - server 把内容 JSON-RPC 编码,通过 stdio pipe 发回
     - adapter 解码成 LangChain ToolMessage 格式
  8. ToolMessage 追加进 state
  9. LLM 下一轮看到内容,继续决策
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 对 LLM 而言**,这一切是 **透明** 的

LLM 看到的就是「一个普通的工具调用 + 返回值」—— **不知道背后有 server / 协议 / pipe 这些细节**。

这就是 **抽象的力量**:LLM 写 prompt 的人 / 设计 agent 的人都不用了解 MCP,只用 LangChain tool 接口。

**这是好事**:工具的 **运行时实现可以任意变**(本地 → 远程、Python → Rust),agent 代码 **不用动**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比 — MCP 像什么?

</div>

### 🅰️ 类比:**HTTP 之于 Web**

**1990 年代前**:每个浏览器要懂每个 server 的 **私有协议**。** 一团乱**。

**HTTP 出现后**:**协议统一**,任何 server 实现 HTTP,任何 client 都能用。

**MCP 之于 AI agent** = **HTTP 之于 web server** —— **标准化让生态爆发**。

### 🅱️ 类比:**JDBC / ODBC 之于数据库**

**没有 JDBC**:Java 程序要直接调 MySQL 的 native driver,换 Oracle 要重写代码。

**有了 JDBC**:**统一接口**,DB 引擎可换,业务代码不用动。

**MCP** = **agent 的 JDBC** —— 后端工具(Filesystem、Slack、PayPal)可换,agent 代码不用动。

### 🆚 对比:**自定义 `@tool`** vs **MCP**

| 维度 | `@tool` | MCP |
|------|---------|-----|
| **快速原型** | ⚡ 写完一个函数就能用 | 🐢 要起 server |
| **跨项目复用** | ❌ 每个项目重写 | ✅ 一处写,共享 |
| **跨语言** | ❌ 只能 Python | ✅ Node/Python/Go 都行 |
| **生态** | ❌ 各自为政 | ✅ Anthropic + 社区维护 |
| **生产部署** | 🟡 工具跟 agent 同进程 | ✅ 工具可独立部署、扩缩容 |
| **适合阶段** | **MVP / 内部工具** | **生产 / 集成 SaaS** |

**结论**:**MVP 先用 `@tool`,需要规模化 / 接 SaaS 时再 MCP**。

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 5 个最容易踩的坑**

1. **❌ 用 `tool.invoke(...)` 而不是 `await tool.ainvoke(...)`** → 报错 / 阻塞。**MCP 必须 async**

2. **❌ node 函数定义成 `def` 而不是 `async def`** → 同上,async 是病毒式传染的(一处 async 全链路 async)

3. **❌ stdio server 启动失败但忘了看错误** → MCP server 是子进程,启动失败时错误在 stderr,**容易被吞** → 用 try/except + 打日志

4. **❌ 远程 server 没设 timeout** → 远程 MCP 可能很慢,**默认 timeout 容易超** → 显式设大

5. **❌ 直接信任远程 MCP server 的输入** → 第三方 server **可能注入恶意内容到 ToolMessage**,LLM 看到后可能被引导做坏事(prompt injection)→ **远程 server 要走白名单 + 内容过滤**

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**MCP = 「**工具的 HTTP**」** —— 标准化让任何 agent 都能用任何工具,**跨进程 / 跨语言 / 跨组织**。

**核心要点**:

- **3 个角色**:LangGraph agent ↔ MCP client(adapter)↔ MCP server
- **2 种 transport**:stdio(本地子进程,快、安全)/ HTTP(远程 server,跨组织)
- **必须 async**(协议 + 子进程通信 + I/O 密集)
- **对 agent 透明**(用 LangChain tool 接口就行)

### 🎯 学完这节,你应该能回答:

1. ✅ **MCP 解决什么问题?**(工具标准化,**跨项目 / 跨语言复用**)
2. ✅ **stdio vs HTTP 怎么选?**(本地用 stdio,SaaS / 跨组织用 HTTP)
3. ✅ **为啥必须 async?**(子进程通信 + I/O 密集 + 并发)
4. ✅ **MultiServerMCPClient 干啥?**(启动 / 管理 server + 协议转 LangChain 格式)
5. ✅ **什么时候用 @tool,什么时候用 MCP?**(MVP 用 @tool,生产 / 集成 SaaS 用 MCP)

### 🚀 2026 实战建议

- 写新 agent 时,**优先看官方 MCP server 有没有现成的**([modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers))
- 公司内部工具,**起一个本地 MCP server 给所有 agent 共用**
- 远程 server,**优先官方(anthropic / asana / cloudflare ...)**,第三方要审计

📎 配套阅读:[`3_research_agent_mcp.ipynb` 主课](./3_research_agent_mcp.ipynb) · 下一节 [`4_z_⭐️_本节精华_Supervisor.ipynb`](./4_z_⭐️_本节精华_Supervisor.ipynb)

</div>